# Register publications in entitycore
Note: Requires admin access atm

In [1]:
import pandas as pd
import os

from datetime import datetime
from entitysdk import Client, ProjectContext, models
from obi_auth import get_token

In [2]:
# Staging environment
token = get_token(environment="staging")
project_context = ProjectContext.from_vlab_url("https://staging.openbraininstitute.org/app/virtual-lab/lab/1f91f30e-1489-4e2a-8eb7-1217257c8e19/project/7a411785-6895-4839-aaa2-d9f76e09875a/home")
client = Client(environment="staging", project_context=project_context, token_manager=token)

# Production environment
# token = get_token(environment="production")
# project_context = ProjectContext.from_vlab_url("https://www.openbraininstitute.org/app/virtual-lab/lab/5f8376bf-b84f-4188-8ef5-e1df3d7529b4/project/7d22829c-edc6-4b1d-8ab9-99dd9e511e74/home")
# client = Client(environment="production", project_context=project_context, token_manager=token)

In [3]:
input_root = "/Users/pokorny/OneDrive - Open Brain Institute/Circuits hardcoded"

In [4]:
check_only = True  # If True, only check, no registration

### Load list of publications to be registered

In [5]:
# sheet_name = "SSCX_literature_new_format"
# sheet_name = "Manipulated_literature_new_form"
# sheet_name = "Hippocampus_literature_new_form"
# sheet_name = "MICrONS_literature_new_format"
# sheet_name = "H01_literature_new_format"
sheet_name = "Allen_V1_literature_new_format"

circuits_file = f"{input_root}/Circuit list.xlsx"
publications = pd.read_excel(circuits_file, sheet_name=sheet_name)
print(f"{publications.shape[0]} publication(s) in table '{sheet_name}'")

15 publication(s) in table 'Allen_V1_literature_new_format'


### Register publications

In [6]:
def remove_multiple_spaces(text):
    split = text.strip().split(" ")
    split = [_txt for _txt in split if len(_txt) > 0]
    return " ".join(split)

def remove_dots(text):
    return text.replace(".", "")

def get_author_list(authors):
    """Split list of authors into individual given/familiy name strings.

    Assuming the following format:
    "FirstName(s)1, FamilyName1; FirstName(s)2, FamiliyName2; ..."
    """
    raw_names_list = remove_dots(authors).split(";")
    names_list = []
    for _name in raw_names_list:
        split = _name.split(",")
        assert len(split) == 2, "ERROR: Wrong format, only one comma expected!"
        author = {"given_name": remove_multiple_spaces(split[0]),
                  "family_name": remove_multiple_spaces(split[1])}
        names_list.append(author)
    return names_list

In [7]:
for idx, row in publications.iterrows():
    author_list = get_author_list(row.authors)

    # Create model
    publ_model = models.Publication(
        DOI=row["doi"].strip(),
        title=row["title"].strip(),
        authors=author_list,
        publication_year=row["publicationDate"].year,
        abstract=row["abstract"].strip(),
    )

    # Register new entity
    res = client.search_entity(entity_type=models.Publication, query={"DOI": publ_model.DOI}).all()
    if len(res) == 0:
        if check_only:
            print(f"***CHECK ONLY*** {publ_model}")
        else:
            registered_publ = client.register_entity(publ_model)
    else:
        print(f"***SKIPPING*** Publication with DOI {publ_model.DOI} already registered")

***SKIPPING*** Publication with DOI 10.1016/j.neuron.2020.01.040 already registered
***CHECK ONLY*** id=None creation_date=None update_date=None created_by=None updated_by=None DOI='10.1038/s41467-017-02718-3' title='Systematic generation of biophysically detailed models for diverse cortical neuron types' authors=[{'given_name': 'Nathan W', 'family_name': 'Gouwens'}, {'given_name': 'Jim', 'family_name': 'Berg'}, {'given_name': 'David', 'family_name': 'Feng'}, {'given_name': 'Staci A', 'family_name': 'Sorensen'}, {'given_name': 'Hongkui', 'family_name': 'Zeng'}, {'given_name': 'Michael J', 'family_name': 'Hawrylycz'}, {'given_name': 'Christof', 'family_name': 'Koch'}, {'given_name': 'Anton', 'family_name': 'Arkhipov'}] publication_year=2018 abstract='The cellular components of mammalian neocortical circuits are diverse, and capturing this diversity in computational models is challenging. Here we report an approach for generating biophysically detailed models of 170 individual neurons in